In [ ]:
# ============================================================
# ⚙️ ENGINE — Run this cell ONCE to load functions & catalogs
# ============================================================

import os, stat, tarfile, time, subprocess, shutil

# ─────────────────────── PATHS ───────────────────────
WORK       = "/kaggle/working"
COMFY      = f"{WORK}/ComfyUI"
MODELS_DIR = f"{COMFY}/models"
VENV       = f"{WORK}/venv"
PY         = f"{VENV}/bin/python"
PIP        = f"{VENV}/bin/pip"
ZROK2      = f"{WORK}/zrok2/zrok2"
TMP        = "/tmp/models"
SNAP_IN    = "/kaggle/input/notebooks/mohamedsalemm11/nb-260221/comfyui_full_snapshot.tar.gz"
SNAP_OUT   = f"{WORK}/comfyui_snapshot.tar.gz"

MODEL_SUBDIRS = [
    "checkpoints", "diffusion_models", "clip", "vae", "unet",
    "controlnet", "loras", "upscale_models", "diffusers",
    "text_encoders", "model_patches",
]

FAST_MODE_LOADED = False

# ─────────────────────── HELPERS ───────────────────────

def _sh(cmd, quiet=False):
    """Run a shell command, optionally print output."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    out = r.stdout + (r.stderr if r.returncode else "")
    if not quiet and out.strip():
        print(out.strip())
    return r.returncode

def _fmt(b):
    if b >= 1024**3: return f"{b/1024**3:.2f} GB"
    if b >= 1024**2: return f"{b/1024**2:.1f} MB"
    return f"{b/1024:.0f} KB"

def _set_env():
    os.environ.update({
        "HF_HOME": "/tmp/huggingface",
        "HF_HUB_CACHE": "/tmp/huggingface/hub",
        "TRANSFORMERS_CACHE": "/tmp/huggingface/transformers",
        "DIFFUSERS_CACHE": "/tmp/huggingface/diffusers",
        "MPLBACKEND": "Agg",
        "TORCH_HOME": "/tmp/torch",
        "XDG_CACHE_HOME": "/tmp/cache",
        "USE_TF": "0",
        "TF_CPP_MIN_LOG_LEVEL": "3",
    })

def _init_dirs():
    _sh("mkdir -p /tmp/huggingface/{hub,diffusers} /tmp/torch /tmp/cache", True)
    _sh(f"mkdir -p {' '.join(f'/tmp/models/{d}' for d in MODEL_SUBDIRS)}", True)
    _sh(f"mkdir -p {MODELS_DIR}", True)
    _sh("apt-get install -y aria2 > /dev/null 2>&1", True)

def _link_models():
    for d in MODEL_SUBDIRS:
        t = f"{MODELS_DIR}/{d}"
        if os.path.islink(t) or os.path.isdir(t):
            _sh(f"rm -rf {t}", True)
        _sh(f"ln -s /tmp/models/{d} {t}", True)
    tmp_m = f"{WORK}/temp-models"
    _sh(f"mkdir -p {tmp_m}", True)
    if not os.path.exists(f"{MODELS_DIR}/checkpoints/temp-models"):
        _sh(f"ln -s {tmp_m} {MODELS_DIR}/checkpoints/", True)

# ─────────────────────── DOWNLOAD ENGINE ───────────────────────

def aria2_dl(url, dest, fname, conn=16):
    os.makedirs(dest, exist_ok=True)
    fp = os.path.join(dest, fname)
    if os.path.exists(fp):
        print(f"  ✅ {fname} ({_fmt(os.path.getsize(fp))}) — cached")
        return True
    t0 = time.time()
    r = subprocess.run([
        "aria2c", "--console-log-level=error", "--summary-interval=0",
        "--download-result=hide", "--show-console-readout=false",
        "--enable-color=false", "-x", str(conn), "-s", str(conn),
        "-k", "1M", "--allow-overwrite=true", "-d", dest, "-o", fname, url
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    dt = time.time() - t0
    if r.returncode == 0 and os.path.exists(fp):
        sz = os.path.getsize(fp)
        print(f"  ✅ {fname} — {_fmt(sz)} in {dt:.1f}s ({sz/1024**2/max(dt,0.1):.1f} MB/s)")
        return True
    print(f"  ❌ {fname} FAILED (code {r.returncode})")
    for ln in (r.stdout or "").strip().splitlines()[-3:]:
        print(f"     {ln}")
    return False

def dl_batch(items, label="Models"):
    if not items:
        print(f"⏭️  {label} — nothing selected")
        return
    print(f"🚀 {label} — {len(items)} file(s)")
    ok = sum(1 for m in items if aria2_dl(m["url"], f'{TMP}/{m["dir"]}', m["file"]))
    fail = len(items) - ok
    print(f"{'✅' if not fail else '⚠️'}  {label} — {ok}/{len(items)} downloaded" +
          (f", {fail} failed" if fail else ""))

# ═══════════════════════════════════════════════════════════
#  MODEL CATALOG  (all unchecked by default except Klein 4B)
#  Each: name, file, url, dir, on (default), gb (approx), grp (UI group)
# ═══════════════════════════════════════════════════════════

SEC_SDXL = [
    # — Checkpoints —
    {"name":"RealisticVisionV6","file":"RealisticVisionV6.safetensors","url":"https://civitai.com/api/download/models/501240","dir":"checkpoints","on":False,"gb":2.0,"grp":"Checkpoints"},
    {"name":"JuggernautXL","file":"JuggernautXL.safetensors","url":"https://civitai.com/api/download/models/1759168","dir":"checkpoints","on":False,"gb":6.5,"grp":"Checkpoints"},
    {"name":"RealVisXL","file":"RealVisXL.safetensors","url":"https://civitai.com/api/download/models/789646","dir":"checkpoints","on":False,"gb":6.5,"grp":"Checkpoints"},
    {"name":"RealVisXL Lightning","file":"RealVisXL_lightning.safetensors","url":"https://civitai.com/api/download/models/798204","dir":"checkpoints","on":False,"gb":6.5,"grp":"Checkpoints"},
    # — ControlNet SD1.5 (~136 MB each) —
    {"name":"CN 1.5 Canny","file":"control_lora_rank128_v11p_sd15_canny_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_canny_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Depth","file":"control_lora_rank128_v11f1p_sd15_depth_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11f1p_sd15_depth_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Inpaint","file":"control_lora_rank128_v11p_sd15_inpaint_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_inpaint_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 IP2P","file":"control_lora_rank128_v11e_sd15_ip2p_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11e_sd15_ip2p_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Lineart","file":"control_lora_rank128_v11p_sd15_lineart_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_lineart_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Lineart Anime","file":"control_lora_rank128_v11p_sd15s2_lineart_anime_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15s2_lineart_anime_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 MLSD","file":"control_lora_rank128_v11p_sd15_mlsd_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_mlsd_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 NormalBAE","file":"control_lora_rank128_v11p_sd15_normalbae_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_normalbae_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 OpenPose","file":"control_lora_rank128_v11p_sd15_openpose_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_openpose_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Scribble","file":"control_lora_rank128_v11p_sd15_scribble_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_scribble_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Seg","file":"control_lora_rank128_v11p_sd15_seg_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_seg_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Shuffle","file":"control_lora_rank128_v11e_sd15_shuffle_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11e_sd15_shuffle_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 SoftEdge","file":"control_lora_rank128_v11p_sd15_softedge_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11p_sd15_softedge_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    {"name":"CN 1.5 Tile","file":"control_lora_rank128_v11f1e_sd15_tile_fp16.safetensors","url":"https://huggingface.co/comfyanonymous/ControlNet-v1-1_fp16_safetensors/resolve/main/control_lora_rank128_v11f1e_sd15_tile_fp16.safetensors","dir":"controlnet","on":False,"gb":0.13,"grp":"ControlNet SD1.5"},
    # — LoRAs —
    {"name":"LCM LoRA SD1.5","file":"lcm-lora-sdv1-5.safetensors","url":"https://huggingface.co/latent-consistency/lcm-lora-sdv1-5/resolve/main/pytorch_lora_weights.safetensors","dir":"loras","on":False,"gb":0.07,"grp":"LoRAs"},
    {"name":"Detail Tweaker","file":"detail_tweaker.safetensors","url":"https://civitai.com/api/download/models/62833","dir":"loras","on":False,"gb":0.14,"grp":"LoRAs"},
    {"name":"My LoRA (epoch 20)","file":"mylora.safetensors","url":"https://download1475.mediafire.com/iy7rbs87t54gDut0MyJpQHnB3LcKcintEVBggzX0E3jNG_v6ivzBii4eS03G1mklt-NAPQ91gvPu6XOp3bsakW17FFxorUmRyZq77JJAwsvQb-XWjFeIaD-X0N2Xn0rRGlX_HwJ5KOmyXTyrqL_pUR7Kq0zCRmtoFktssIdh8BA/kq7p6thopglgydk/20251225-1766670376063-000020.safetensors","dir":"loras","on":False,"gb":0.15,"grp":"LoRAs"},
    {"name":"My LoRA (epoch 19)","file":"mylora19.safetensors","url":"https://download947.mediafire.com/a1wanpeyi03gm4oGTpJTu4FtEBkIpSpLKcD_ceup1cTuJ1o-xr0VNaaPa7IuUix5LxmhObuueadkAT8ce2oMDZrPwLyt8INGCoq2y9DeKQd6EZ0a_mtYHeCRAqdKTE59L_VRa3db-NEvo5RUdFn4uEFMi3p1gYxvprW7b4mUnSo/icjf2295m51xnkh/20251225-1766670376063-000019.safetensors","dir":"loras","on":False,"gb":0.15,"grp":"LoRAs"},
    # — ControlNet SDXL: Kohya LLLite (lightweight) —
    {"name":"Kohya XL Canny","file":"controllllite_v01032064e_sdxl_canny.safetensors","url":"https://huggingface.co/kohya-ss/controlnet-lllite/resolve/main/controllllite_v01032064e_sdxl_canny.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    {"name":"Kohya XL Depth","file":"controllllite_v01032064e_sdxl_depth.safetensors","url":"https://huggingface.co/kohya-ss/controlnet-lllite/resolve/main/controllllite_v01032064e_sdxl_depth.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    {"name":"Kohya XL OpenPose","file":"controllllite_v01032064e_sdxl_openpose.safetensors","url":"https://huggingface.co/kohya-ss/controlnet-lllite/resolve/main/controllllite_v01032064e_sdxl_openpose.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    {"name":"Kohya XL Blur","file":"controllllite_v01032064e_sdxl_blur.safetensors","url":"https://huggingface.co/kohya-ss/controlnet-lllite/resolve/main/controllllite_v01032064e_sdxl_blur.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    {"name":"Kohya XL Scribble","file":"controllllite_v01032064e_sdxl_scribble.safetensors","url":"https://huggingface.co/kohya-ss/controlnet-lllite/resolve/main/controllllite_v01032064e_sdxl_scribble.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    {"name":"Kohya XL Tile Realistic","file":"bdsqlsz_controlllite_xl_tile_realistic.safetensors","url":"https://huggingface.co/bdsqlsz/qinglong_controlnet-lllite/resolve/main/bdsqlsz_controlllite_xl_tile_realistic.safetensors","dir":"controlnet","on":False,"gb":0.05,"grp":"CN SDXL Kohya"},
    # — ControlNet SDXL: Stability AI Control-LoRA —
    {"name":"SAI XL Canny 128","file":"sai_xl_canny_128lora.safetensors","url":"https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank128/control-lora-canny-rank128.safetensors","dir":"controlnet","on":False,"gb":0.35,"grp":"CN SDXL SAI"},
    {"name":"SAI XL Depth 128","file":"sai_xl_depth_128lora.safetensors","url":"https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank128/control-lora-depth-rank128.safetensors","dir":"controlnet","on":False,"gb":0.35,"grp":"CN SDXL SAI"},
    {"name":"SAI XL Sketch 128","file":"sai_xl_sketch_128lora.safetensors","url":"https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank128/control-lora-sketch-rank128.safetensors","dir":"controlnet","on":False,"gb":0.35,"grp":"CN SDXL SAI"},
    {"name":"SAI XL Recolor 128","file":"sai_xl_recolor_128lora.safetensors","url":"https://huggingface.co/stabilityai/control-lora/resolve/main/control-LoRAs-rank128/control-lora-recolor-rank128.safetensors","dir":"controlnet","on":False,"gb":0.35,"grp":"CN SDXL SAI"},
    # — ControlNet SDXL: xinsir (full-sized, high quality) —
    {"name":"xinsir XL Tile","file":"controlnet-tile-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-tile-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    {"name":"xinsir XL Union","file":"controlnet-union-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-union-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    {"name":"xinsir XL Depth","file":"controlnet-depth-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    {"name":"xinsir XL Canny","file":"controlnet-canny-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    {"name":"xinsir XL OpenPose","file":"controlnet-openpose-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-openpose-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    {"name":"xinsir XL Scribble","file":"controlnet-scribble-sdxl-1.0.safetensors","url":"https://huggingface.co/xinsir/controlnet-scribble-sdxl-1.0/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL xinsir"},
    # — ControlNet SDXL: Diffusers fp16 —
    {"name":"Diffusers XL Canny fp16","file":"diffusers_xl_canny_full.safetensors","url":"https://huggingface.co/diffusers/controlnet-canny-sdxl-1.0/resolve/main/diffusion_pytorch_model.fp16.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL Diffusers"},
    {"name":"Diffusers XL Depth fp16","file":"diffusers_xl_depth_full.safetensors","url":"https://huggingface.co/diffusers/controlnet-depth-sdxl-1.0/resolve/main/diffusion_pytorch_model.fp16.safetensors","dir":"controlnet","on":False,"gb":2.5,"grp":"CN SDXL Diffusers"},
    # — Embeddings —
    {"name":"EasyNegative","file":"easynegative.safetensors","url":"https://civitai.com/api/download/models/9208","dir":"embeddings","on":False,"gb":0.01,"grp":"Embeddings"},
    # — Upscalers —
    {"name":"4x Foolhardy Remacri","file":"4x_foolhardy_Remacri.pth","url":"https://huggingface.co/FacehugmanIII/4x_foolhardy_Remacri/resolve/main/4x_foolhardy_Remacri.pth","dir":"upscale_models","on":False,"gb":0.07,"grp":"Upscalers"},
]

SEC_ZIMAGE = [
    {"name":"Z-Image Turbo Q8 GGUF","file":"z_image_turbo-Q8_0.gguf","url":"https://huggingface.co/jayn7/Z-Image-Turbo-GGUF/resolve/main/z_image_turbo-Q8_0.gguf","dir":"diffusion_models","on":False,"gb":7.2,"grp":"Core"},
    {"name":"Qwen3-4B Text Enc Q8","file":"Qwen3-4B-Q8_0.gguf","url":"https://huggingface.co/unsloth/Qwen3-4B-GGUF/resolve/main/Qwen3-4B-Q8_0.gguf","dir":"text_encoders","on":False,"gb":4.5,"grp":"Core"},
    {"name":"VAE (ae.safetensors)","file":"ae.safetensors","url":"https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors","dir":"vae","on":False,"gb":0.34,"grp":"Core"},
]

SEC_FLUX2DEV = [
    {"name":"Flux.2 Dev UNet Q8","file":"flux2-dev-Q8_0.gguf","url":"https://huggingface.co/city96/FLUX.2-dev-gguf/resolve/main/flux2-dev-Q8_0.gguf","dir":"unet","on":False,"gb":35.0,"grp":"Core"},
    {"name":"T5-XXL Encoder Q8","file":"t5-v1_1-xxl-encoder-Q8_0.gguf","url":"https://huggingface.co/UmeAiRT/ComfyUI-Auto_installer/resolve/main/models/clip/t5-v1_1-xxl-encoder-Q8_0.gguf","dir":"clip","on":False,"gb":9.5,"grp":"Core"},
    {"name":"ViT-L-14 CLIP","file":"ViT-L-14-TEXT-detail-improved-hiT-GmP-TE-only-HF.safetensors","url":"https://huggingface.co/UmeAiRT/ComfyUI-Auto_installer/resolve/main/models/clip/ViT-L-14-TEXT-detail-improved-hiT-GmP-TE-only-HF.safetensors","dir":"clip","on":False,"gb":0.25,"grp":"Core"},
    {"name":"Flux VAE","file":"ae.safetensors","url":"https://huggingface.co/UmeAiRT/ComfyUI-Auto_installer/resolve/main/models/vae/ae.safetensors","dir":"vae","on":False,"gb":0.34,"grp":"Core"},
    {"name":"RealESRGAN x4","file":"RealESRGAN_x4plus.pth","url":"https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth","dir":"upscale_models","on":False,"gb":0.06,"grp":"Upscaler"},
]

SEC_QWEN = [
    {"name":"Qwen Image Diff fp8","file":"qwen_image_fp8_e4m3fn.safetensors","url":"https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors","dir":"diffusion_models","on":False,"gb":12.0,"grp":"Core"},
    {"name":"Qwen 2.5-VL 7B fp8","file":"qwen_2.5_vl_7b_fp8_scaled.safetensors","url":"https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors","dir":"text_encoders","on":False,"gb":8.0,"grp":"Core"},
    {"name":"Qwen Image VAE","file":"qwen_image_vae.safetensors","url":"https://huggingface.co/Comfy-Org/Qwen-Image_ComfyUI/resolve/main/split_files/vae/qwen_image_vae.safetensors","dir":"vae","on":False,"gb":0.34,"grp":"Core"},
    {"name":"Qwen InstantX Union CN","file":"Qwen-Image-InstantX-ControlNet-Union.safetensors","url":"https://huggingface.co/InstantX/Qwen-Image-ControlNet-Union/resolve/main/Qwen-Image-InstantX-ControlNet-Union.safetensors","dir":"controlnet","on":False,"gb":4.5,"grp":"ControlNet"},
]

SEC_KREA = [
    {"name":"Flux Krea Dev Q8","file":"flux1-krea-dev-Q8_0.gguf","url":"https://huggingface.co/QuantStack/FLUX.1-Krea-dev-GGUF/resolve/main/flux1-krea-dev-Q8_0.gguf","dir":"unet","on":False,"gb":12.0,"grp":"Core"},
    {"name":"Flux VAE (ae)","file":"ae.safetensors","url":"https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors","dir":"vae","on":False,"gb":0.34,"grp":"Core"},
    {"name":"CLIP-L","file":"clip_l.safetensors","url":"https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors","dir":"clip","on":False,"gb":0.25,"grp":"Core"},
    {"name":"T5-XXL Enc Q8","file":"t5-v1_1-xxl-encoder-Q8_0.gguf","url":"https://huggingface.co/city96/t5-v1_1-xxl-encoder-gguf/resolve/main/t5-v1_1-xxl-encoder-Q8_0.gguf","dir":"clip","on":False,"gb":9.5,"grp":"Core"},
]

SEC_KLEIN = [
    {"name":"Flux 2 VAE","file":"flux2-vae.safetensors","url":"https://huggingface.co/Comfy-Org/flux2-dev/resolve/main/split_files/vae/flux2-vae.safetensors","dir":"vae","on":True,"gb":0.34,"grp":"Base"},
    {"name":"Klein 4B UNet Q8","file":"flux-2-klein-4b-Q8_0.gguf","url":"https://huggingface.co/unsloth/FLUX.2-klein-4B-GGUF/resolve/main/flux-2-klein-4b-Q8_0.gguf","dir":"unet/gguf","on":True,"gb":4.3,"grp":"Klein 4B"},
    {"name":"Klein 4B Text Enc","file":"qwen_3_4b.safetensors","url":"https://huggingface.co/Comfy-Org/flux2-klein-4B/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors","dir":"text_encoders","on":True,"gb":8.0,"grp":"Klein 4B"},
    {"name":"Klein 9B UNet Q8","file":"flux-2-klein-9b-Q8_0.gguf","url":"https://huggingface.co/unsloth/FLUX.2-klein-9B-GGUF/resolve/main/flux-2-klein-9b-Q8_0.gguf","dir":"unet/gguf","on":False,"gb":10.0,"grp":"Klein 9B"},
    {"name":"Klein 9B Text Enc","file":"qwen_3_8b.safetensors","url":"https://huggingface.co/Comfy-Org/flux2-klein-9B/resolve/main/split_files/text_encoders/qwen_3_8b.safetensors","dir":"text_encoders","on":False,"gb":16.0,"grp":"Klein 9B"},
    {"name":"InstantX Union CN","file":"diffusion_pytorch_model.safetensors","url":"https://huggingface.co/InstantX/FLUX.1-dev-Controlnet-Union/resolve/main/diffusion_pytorch_model.safetensors","dir":"controlnet/InstantX-Union","on":False,"gb":3.5,"grp":"ControlNet"},
]

ALL_SECTIONS = [
    ("sdxl",      "SDXL / SD1.5 + LoRA + ControlNet",  SEC_SDXL),
    ("z_image",   "Z-Image Turbo (~12 GB)",             SEC_ZIMAGE),
    ("flux2_dev", "Flux.2 Dev Q8 (~48 GB)",             SEC_FLUX2DEV),
    ("qwen",      "Qwen Image (~25 GB)",                SEC_QWEN),
    ("krea",      "Flux Krea Q8 (~22 GB)",              SEC_KREA),
    ("klein",     "Flux 2 Klein (~5-30 GB)",            SEC_KLEIN),
]

# ═══════════════════════════════════════════════════════════
#  CUSTOM NODES CATALOG
# ═══════════════════════════════════════════════════════════

NODES_CATALOG = [
    # — Active by default —
    {"name":"ComfyUI Essentials","url":"https://github.com/cubiq/ComfyUI_essentials.git","on":True},
    {"name":"ComfyUI MultiGPU","url":"https://github.com/pollockjj/ComfyUI-MultiGPU.git","on":True},
    {"name":"Comfyroll Custom Nodes","url":"https://github.com/Suzie1/ComfyUI_Comfyroll_CustomNodes.git","on":True},
    {"name":"ComfyUI Easy-Use","url":"https://github.com/yolain/ComfyUI-Easy-Use.git","on":True},
    {"name":"ComfyUI GGUF","url":"https://github.com/city96/ComfyUI-GGUF.git","on":True},
    {"name":"rgthree-comfy","url":"https://github.com/rgthree/rgthree-comfy.git","on":True},
    {"name":"ComfyUI KJ Nodes","url":"https://github.com/kijai/ComfyUI-KJNodes.git","on":True},
    {"name":"ComfyUI Custom Scripts","url":"https://github.com/pythongosssss/ComfyUI-Custom-Scripts.git","on":True},
    {"name":"ComfyUI Crystools","url":"https://github.com/crystian/ComfyUI-Crystools.git","on":True},
    {"name":"ControlAltAI Nodes","url":"https://github.com/gseth/ControlAltAI-Nodes","on":True},
    {"name":"cg-use-everywhere","url":"https://github.com/chrisgoringe/cg-use-everywhere.git","on":True},
    {"name":"rembg ComfyUI Node","url":"https://github.com/Jcd1230/rembg-comfyui-node.git","on":True},
    {"name":"ComfyUI Angelo","url":"https://github.com/shootthesound/ComfyUI-Angelo","on":True},
    {"name":"ComfyUI Gallery","url":"https://github.com/PanicTitan/ComfyUI-Gallery.git","on":True},
    {"name":"Impact Pack","url":"https://github.com/ltdrdata/ComfyUI-Impact-Pack","on":True},
    {"name":"ControlNet AUX","url":"https://github.com/Fannovel16/comfyui_controlnet_aux.git","on":True},
    {"name":"IPAdapter Plus","url":"https://github.com/cubiq/ComfyUI_IPAdapter_plus.git","on":True},
    {"name":"Inpaint Nodes","url":"https://github.com/Acly/comfyui-inpaint-nodes.git","on":True},
    {"name":"ComfyUI Gallery","url":"https://github.com/PanicTitan/ComfyUI-Gallery.git","on":True},
    {"name":"ComfyUI Gallery","url":"https://github.com/PanicTitan/ComfyUI-Gallery.git","on":True},
    {"name":"comfyui-inpaint-nodes","url":"https://github.com/Acly/comfyui-inpaint-nodes.git","on":True},
    {"name":"comfyui-art-venture","url":"https://github.com/sipherxyz/comfyui-art-venture.git","on":True},
    {"name":"ComfyUI-Florence2","url":"https://github.com/kijai/ComfyUI-Florence2.git","on":True},
    {"name":"ComfyUI-SAM3","url":"https://github.com/PozzettiAndrea/ComfyUI-SAM3","on":True},
    {"name":"rembg-comfyui-node","url":"https://github.com/Jcd1230/rembg-comfyui-node.git","on":True},
    {"name":"ComfyUI-SeedVR2_VideoUpscaler","url":"https://github.com/numz/ComfyUI-SeedVR2_VideoUpscaler.git","on":True},
    {"name":"ComfyUI_UltimateSDUpscale","url":"https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git","on":True},


    
    # — Optional (off by default) —
    {"name":"ComfyUI FluxTrainer","url":"https://github.com/1038lab/ComfyUI-RMBG","on":False},
    {"name":"ComfyUI to Python Ext","url":"https://github.com/pydn/ComfyUI-to-Python-Extension.git","on":False},
    {"name":"Inpaint CropAndStitch","url":"https://github.com/lquesada/ComfyUI-Inpaint-CropAndStitch","on":False},
    {"name":"ComfyUI Impact Pack","url":"https://github.com/ltdrdata/ComfyUI-Impact-Pack","on":False},
    {"name":"WAS Node Suite","url":"https://github.com/WASasquatch/was-node-suite-comfyui","on":False},
    {"name":"ComfyUI Inspire Pack","url":"https://github.com/ltdrdata/ComfyUI-Inspire-Pack","on":False},
    {"name":"ComfyUI N-Sidebar","url":"https://github.com/Nuked88/ComfyUI-N-Sidebar.git","on":False},
    {"name":"Dynamic Prompts","url":"https://github.com/adieyal/comfyui-dynamicprompts","on":False},
    {"name":"Styles CSV Loader","url":"https://github.com/theUpsider/ComfyUI-Styles_CSV_Loader.git","on":False},
    {"name":"SDXL Prompt Styler","url":"https://github.com/twri/sdxl_prompt_styler.git","on":False},
    {"name":"ComfyUI IF AI Tools","url":"https://github.com/if-ai/ComfyUI-IF_AI_tools.git","on":False},
    {"name":"ControlNet Aux","url":"https://github.com/Fannovel16/comfyui_controlnet_aux.git","on":False},
    {"name":"IPAdapter Plus","url":"https://github.com/cubiq/ComfyUI_IPAdapter_plus.git","on":False},
    {"name":"ComfyUI LayerStyle","url":"https://github.com/chflame163/ComfyUI_LayerStyle.git","on":False},
    {"name":"Inpaint Nodes","url":"https://github.com/Acly/comfyui-inpaint-nodes.git","on":False},
    {"name":"Art Venture","url":"https://github.com/sipherxyz/comfyui-art-venture.git","on":False},
    {"name":"ComfyUI Florence2","url":"https://github.com/kijai/ComfyUI-Florence2.git","on":False},
    {"name":"ComfyUI RMBG","url":"https://github.com/1038lab/ComfyUI-RMBG.git","on":False},
    {"name":"Segment Anything 2","url":"https://github.com/kijai/ComfyUI-segment-anything-2.git","on":False},
    {"name":"ComfyUI 3D Pack","url":"https://github.com/MrForExample/ComfyUI-3D-Pack.git","on":False},
    {"name":"ComfyUI Lotus","url":"https://github.com/kijai/ComfyUI-Lotus.git","on":False},
    {"name":"Tooling Nodes","url":"https://github.com/Acly/comfyui-tooling-nodes.git","on":False},
    {"name":"Ultimate SD Upscale","url":"https://github.com/ssitu/ComfyUI_UltimateSDUpscale.git","on":False},
    {"name":"Relight","url":"https://github.com/EnragedAntelope/comfyui-relight.git","on":False},
    {"name":"IC-Light","url":"https://github.com/huchenlei/ComfyUI-IC-Light","on":False},
    {"name":"HQ Image Save","url":"https://github.com/spacepxl/ComfyUI-HQ-Image-Save.git","on":False},
    {"name":"Enricos Nodes","url":"https://github.com/erosDiffusion/ComfyUI-enricos-nodes.git","on":False},
    {"name":"PoP Nodes","url":"https://github.com/picturesonpictures/comfy_PoP.git","on":False},
    {"name":"Image Filters","url":"https://github.com/spacepxl/ComfyUI-Image-Filters.git","on":False},
    {"name":"AutomaticCFG","url":"https://github.com/Extraltodeus/ComfyUI-AutomaticCFG.git","on":False},
    {"name":"ReActor","url":"https://github.com/Gourieff/ComfyUI-ReActor","on":False},
]

# Count defaults
_n_active = sum(1 for n in NODES_CATALOG if n["on"])
_n_total  = len(NODES_CATALOG)

# ═══════════════════════════════════════════════════════════
#  OPERATION FUNCTIONS
# ═══════════════════════════════════════════════════════════

def do_extract_snapshot():
    """Extract snapshot (returning users)."""
    global FAST_MODE_LOADED
    print("⚡ Fast restore from snapshot...")
    if not os.path.exists(SNAP_IN):
        print(f"❌ Missing: {SNAP_IN}")
        print("👉 If first time, use 'Full Setup' instead.")
        if os.path.exists("/kaggle/input"):
            print("📋 /kaggle/input:", ", ".join(sorted(os.listdir("/kaggle/input"))))
        return False

    t0 = time.time()
    print(f"📦 Found snapshot: {os.path.getsize(SNAP_IN)/(1024**3):.2f} GB")
    with tarfile.open(SNAP_IN, "r:gz") as tar:
        tar.extractall(WORK)
    print(f"✅ Extracted in {time.time()-t0:.1f}s")

    # Auto-repair venv
    repair = f"{WORK}/_venv_repair.py"
    if os.path.exists(repair):
        subprocess.run(["python3", repair], capture_output=True, text=True)
        os.remove(repair)

    # Fix permissions
    for bp in [f"{VENV}/bin", f"{WORK}/zrok2"]:
        if os.path.exists(bp):
            for fn in os.listdir(bp):
                fp = os.path.join(bp, fn)
                if os.path.isfile(fp):
                    os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

    # Fix python symlinks
    if not os.path.exists(f"{VENV}/bin/python3.10"):
        _sh(f"cp /usr/bin/python3.10 {VENV}/bin/", True)
    _sh(f"ln -sf {VENV}/bin/python3.10 {VENV}/bin/python", True)
    _sh(f"ln -sf {VENV}/bin/python3.10 {VENV}/bin/python3", True)

    _set_env(); _init_dirs(); _link_models()
    
    ok = lambda p: "✅" if os.path.exists(p) else "❌"
    print(f"✅ Restored: ComfyUI {ok(COMFY)} | venv {ok(VENV)} | zrok2 {ok(ZROK2)} | py {ok(PY)} | {time.time()-t0:.1f}s")
    FAST_MODE_LOADED = True
    return True


def do_full_setup():
    """Full first-time setup: ComfyUI + venv + zrok2."""
    global FAST_MODE_LOADED
    if FAST_MODE_LOADED:
        print("⏭️  Fast Mode already loaded; skipping full setup.")
        return True

    print("🔧 Starting Full Setup (this takes ~15 minutes)...")
    os.chdir(WORK)

    # Virtualenv
    _sh("pip install virtualenv", True)
    if not os.path.exists(f"{WORK}/venv"):
        print("📦 Creating virtualenv...")
        _sh(f"virtualenv venv -p $(which python3.10)")
        print("📦 Installing PyTorch 2.6...")
        _sh(f"{PIP} install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
        _sh(f'{PY} -c "import torch; print(\'PyTorch:\', torch.__version__, \'| CUDA:\', torch.cuda.is_available())"')
        print("📦 Installing TensorFlow + deps...")
        _sh(f"{PIP} install tensorflow[and-cuda] safetensors")
        _sh(f"wget -q https://q4j3.c11.e2-5.dev/downloads/req.txt -O {WORK}/req.txt")
        _sh(f"{PIP} install -r {WORK}/req.txt")
    else:
        for root, _, files in os.walk(f"{VENV}/bin"):
            for fn in files:
                fp = os.path.join(root, fn)
                try: os.chmod(fp, os.stat(fp).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
                except: pass

    # Python symlinks
    if not os.path.exists(f"{VENV}/bin/python3.10"):
        _sh(f"cp /usr/bin/python3.10 {VENV}/bin/", True)
    _sh(f"[ -e {VENV}/bin/python ] || ln -s {VENV}/bin/python3.10 {VENV}/bin/python", True)
    _sh(f"[ -e {VENV}/bin/python3 ] || ln -s {VENV}/bin/python3.10 {VENV}/bin/python3", True)

    # Clone ComfyUI
    os.chdir(WORK)
    if not os.path.exists(COMFY):
        print("📦 Cloning ComfyUI...")
        _sh("git clone https://github.com/comfyanonymous/ComfyUI.git")
    os.chdir(COMFY)
    _sh("git checkout v0.11.0")
    _sh(f"{PIP} install -r requirements.txt")

    # aria2 + zrok2
    print("📦 Installing tools...")
    _sh("apt-get update -y && apt-get install -y aria2", True)
    if not os.path.exists(f"{WORK}/zrok2/zrok2"):
        cmds = (f"mkdir -p {WORK}/zrok2 && cd {WORK}/zrok2 && rm -f zrok2*.gz && "
                f"wget -q https://github.com/openziti/zrok/releases/download/v2.0.0-rc5/zrok_2.0.0-rc5_linux_amd64.tar.gz && "
                f"tar -xf ./zrok2*.gz && chmod a+x {WORK}/zrok2/zrok2")
        _sh(cmds)

    _set_env(); _init_dirs(); _link_models()

    # ComfyUI-Manager + extras
    nodes = f"{COMFY}/custom_nodes"
    os.chdir(nodes)
    if not os.path.exists(f"{nodes}/ComfyUI-Manager"):
        _sh("git clone https://github.com/ltdrdata/ComfyUI-Manager.git")
    os.chdir(f"{nodes}/ComfyUI-Manager"); _sh("git pull", True)
    _sh(f"wget -nc -q https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O {WORK}/pinggy.py")
    os.chdir(nodes)
    if not os.path.exists(f"{nodes}/ComfyBootlegOffload.py"):
        _sh("wget -q https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py")

    # Extra packages
    print("📦 Installing extra packages...")
    _sh(f"{PIP} install -r {COMFY}/requirements.txt -q")
    _sh(f"{PIP} install piexif gguf spandrel -q")
    _sh(f'{PIP} install "rembg[gpu]" -q')
    _sh(f"{PIP} install rembg -q")

    FAST_MODE_LOADED = True
    print("✅ FULL SETUP COMPLETE!")
    return True


def _zrok_list_names(namespace="public"):
    """Best-effort parser for zrok2 list names output."""
    try:
        r = subprocess.run([ZROK2, "list", "names"], capture_output=True, text=True, timeout=30)
        if r.returncode != 0:
            return []
        names = []
        for ln in (r.stdout or "").splitlines():
            s = ln.strip()
            if not s or s.lower().startswith("url") or s.startswith("-"):
                continue
            p = s.split()
            if len(p) < 2:
                continue

            n = None
            ns = None
            if p[0].startswith("http") and len(p) >= 3:
                n, ns = p[1], p[2]
            elif ":" in p[0]:
                ns, n = p[0].split(":", 1)
            elif len(p) >= 3:
                n, ns = p[0], p[1]

            if n and ns == namespace and n not in names:
                names.append(n)
        return names
    except Exception:
        return []


def do_enable_zrok(token, namespace="public", name="", reserved_action="Use Existing"):
    """Enable zrok2 and manage reserved names (use existing or delete/recreate)."""
    if not token.strip():
        print("❌ Please enter your zrok2 token!")
        return ""
    if not os.path.exists(ZROK2):
        print("❌ zrok2 binary not found. Run Setup first.")
        return ""

    requested = name.strip()
    _sh(f"chmod a+x {ZROK2}", True)

    # Re-enable safely (ignore disable failures when already disabled)
    print("🔗 Refreshing zrok2 environment...")
    _sh(f"{ZROK2} disable", True)
    _sh(f"{ZROK2} enable {token}")

    existing = _zrok_list_names(namespace)
    chosen = ""

    if existing:
        print(f"📌 Existing reserved names in '{namespace}': {', '.join(existing)}")
        if reserved_action.startswith("Delete Existing"):
            if not requested:
                print("⚠️ No new name provided, so existing names were kept.")
                chosen = existing[0]
            else:
                for n in existing:
                    if n != requested:
                        _sh(f"{ZROK2} delete name -n {namespace} {n} || true", True)
                        print(f"  🗑️ Deleted old reserved name: {n}")
                if requested not in existing:
                    _sh(f"{ZROK2} create name -n {namespace} {requested}")
                    print(f"  ✅ Created reserved name: {requested}")
                chosen = requested
        else:
            # Use existing if present, else create requested
            if requested and requested in existing:
                chosen = requested
            elif existing:
                chosen = existing[0]
                if requested and requested != chosen:
                    print(f"ℹ️ Using existing reserved name '{chosen}' (requested '{requested}').")
    else:
        if requested:
            _sh(f"{ZROK2} create name -n {namespace} {requested}")
            print(f"✅ Created reserved name: {requested}")
            chosen = requested

    if chosen:
        print(f"✅ Zrok2 enabled. Active reserved name: {namespace}:{chosen}")
    else:
        print("✅ Zrok2 enabled. No reserved name selected (ephemeral sharing will be used).")
    return chosen


# Friendly model-type labels → actual subfolder in /tmp/models/
MODEL_TYPE_MAP = {
    "Checkpoint":        "checkpoints",
    "LoRA":              "loras",
    "VAE":               "vae",
    "Text Encoder":      "text_encoders",
    "CLIP":              "clip",
    "UNet / Diffusion":  "unet",
    "ControlNet":        "controlnet",
    "Upscaler":          "upscale_models",
    "Embedding":         "embeddings",
    "Diffusion Model":   "diffusion_models",
}


def do_download_models(selected):
    """Download selected model files."""
    dl_batch(selected, "Selected Models")


def do_download_custom_model(url, filename, model_type):
    """Download a single model file by URL into the correct ComfyUI folder."""
    url = url.strip()
    filename = filename.strip()
    if not url:
        print("❌ Please enter a download URL!")
        return False
    if not url.startswith("http"):
        print("❌ Invalid URL — must start with https://")
        return False

    # Auto-detect filename from URL if not provided
    if not filename:
        filename = url.rstrip("/").split("/")[-1].split("?")[0]
        if not filename or "." not in filename:
            print("❌ Couldn't detect filename from URL — please enter one manually.")
            return False
        print(f"  ℹ️  Auto-detected filename: {filename}")

    subfolder = MODEL_TYPE_MAP.get(model_type)
    if not subfolder:
        print(f"❌ Unknown model type: {model_type}")
        return False

    dest = f"{TMP}/{subfolder}"
    print(f"📥 Downloading: {filename}")
    print(f"   Type: {model_type} → {subfolder}/")
    print(f"   URL:  {url[:80]}{'…' if len(url) > 80 else ''}")
    ok = aria2_dl(url, dest, filename)
    if ok:
        print(f"✅ {filename} ready in ComfyUI/{subfolder}/")
    return ok


def _tmp_root():
    return os.path.realpath(TMP)


def _safe_tmp_path(rel_path):
    root = _tmp_root()
    rel = str(rel_path or "").strip().replace('\\', '/')
    rel = rel.lstrip('/')
    if not rel:
        raise ValueError("Empty path")
    cand = os.path.realpath(os.path.join(root, rel))
    if os.path.commonpath([root, cand]) != root:
        raise ValueError("Path outside /tmp/models")
    return cand, rel


def _human_mb(num_bytes):
    return f"{num_bytes / (1024**2):.1f} MB"


def _infer_model_type(rel_path):
    rel = rel_path.replace('\\', '/')
    for t, sub in MODEL_TYPE_MAP.items():
        if rel == sub or rel.startswith(sub + '/'):
            return t
    return "Other"


def scan_tmp_models():
    """Return discovered files/folders under /tmp/models for UI management."""
    root = _tmp_root()
    items = []
    if not os.path.exists(root):
        return items

    for dp, dn, fn in os.walk(root):
        for name in fn:
            abs_p = os.path.join(dp, name)
            rel = os.path.relpath(abs_p, root).replace('\\', '/')
            try:
                sz = os.path.getsize(abs_p)
            except OSError:
                sz = 0
            items.append({
                "rel": rel,
                "name": name,
                "type": _infer_model_type(rel),
                "bytes": sz,
            })

    for dp, dn, fn in os.walk(root):
        for name in dn:
            abs_p = os.path.join(dp, name)
            rel = os.path.relpath(abs_p, root).replace('\\', '/')
            if rel in ("", "."):
                continue
            # Include non-empty custom directories (e.g., diffusers bundles)
            try:
                has_content = any(os.scandir(abs_p))
            except OSError:
                has_content = False
            if has_content:
                items.append({
                    "rel": rel,
                    "name": name + "/",
                    "type": _infer_model_type(rel),
                    "bytes": 0,
                    "is_dir": True,
                })

    # De-duplicate and sort
    seen = set()
    uniq = []
    for x in items:
        if x["rel"] in seen:
            continue
        seen.add(x["rel"])
        uniq.append(x)
    uniq.sort(key=lambda x: (x.get("type", ""), x.get("rel", "")))
    return uniq


def delete_tmp_models(rel_paths):
    """Permanently delete selected files/folders from /tmp/models."""
    if not rel_paths:
        print("⏭️  No files selected.")
        return {"deleted": 0, "missing": 0, "failed": 0}

    out = {"deleted": 0, "missing": 0, "failed": 0}
    print(f"🗑️ Deleting {len(rel_paths)} item(s)...")
    for rel in rel_paths:
        try:
            abs_p, safe_rel = _safe_tmp_path(rel)
            if not os.path.exists(abs_p):
                out["missing"] += 1
                print(f"  ⚪ Missing: {safe_rel}")
                continue
            if os.path.isdir(abs_p):
                shutil.rmtree(abs_p)
            else:
                os.remove(abs_p)
            out["deleted"] += 1
            print(f"  ✅ Deleted: {safe_rel}")
        except Exception as e:
            out["failed"] += 1
            print(f"  ❌ Failed: {rel} ({e})")

    print(f"✅ Delete done — {out['deleted']} deleted, {out['missing']} missing, {out['failed']} failed")
    return out


def move_tmp_models(rel_paths, target_type):
    """Move selected files/folders to another model-type folder inside /tmp/models."""
    if not rel_paths:
        print("⏭️  No files selected.")
        return {"moved": 0, "skipped": 0, "failed": 0}

    target_sub = MODEL_TYPE_MAP.get(target_type)
    if not target_sub:
        print(f"❌ Unknown target type: {target_type}")
        return {"moved": 0, "skipped": 0, "failed": len(rel_paths)}

    root = _tmp_root()
    target_dir = os.path.join(root, target_sub)
    os.makedirs(target_dir, exist_ok=True)

    out = {"moved": 0, "skipped": 0, "failed": 0}
    print(f"📦 Moving {len(rel_paths)} item(s) to {target_type}...")
    for rel in rel_paths:
        try:
            src, safe_rel = _safe_tmp_path(rel)
            if not os.path.exists(src):
                out["skipped"] += 1
                print(f"  ⚪ Missing: {safe_rel}")
                continue
            dst = os.path.join(target_dir, os.path.basename(src))
            if os.path.realpath(src) == os.path.realpath(dst):
                out["skipped"] += 1
                print(f"  ↪ Already in target: {safe_rel}")
                continue
            if os.path.exists(dst):
                out["skipped"] += 1
                print(f"  ⚠️ Skipped (exists): {os.path.basename(dst)}")
                continue
            shutil.move(src, dst)
            out["moved"] += 1
            print(f"  ✅ Moved: {safe_rel} → {target_sub}/{os.path.basename(dst)}")
        except Exception as e:
            out["failed"] += 1
            print(f"  ❌ Failed: {rel} ({e})")

    print(f"✅ Move done — {out['moved']} moved, {out['skipped']} skipped, {out['failed']} failed")
    return out


def do_install_nodes(repos):
    """Clone + install dependencies for selected custom nodes."""
    if not repos:
        print("⏭️  No custom nodes selected.")
        return
    nodes_dir = f"{COMFY}/custom_nodes"
    os.makedirs(nodes_dir, exist_ok=True)
    os.chdir(nodes_dir)

    pip_exec = PIP if os.path.exists(PY) else "pip3"
    try:
        if os.path.exists(PY):
            r = subprocess.run([PY, "--version"], capture_output=True, text=True, timeout=10)
            if r.returncode != 0: pip_exec = "pip3"
    except: pip_exec = "pip3"

    print(f"📦 Installing {len(repos)} custom node(s)...")
    cloned, existed, failed_c = 0, 0, []
    for i, url in enumerate(repos, 1):
        rn = url.split("/")[-1].replace(".git", "")
        if os.path.exists(rn):
            existed += 1
        else:
            r = subprocess.run(["git", "clone", url, "-q"], capture_output=True, text=True, timeout=180)
            if r.returncode == 0: cloned += 1
            else: failed_c.append(rn)
        print(f"\r  Cloning: {int(i/len(repos)*100)}% ({i}/{len(repos)})", end="", flush=True)
    print()
    print(f"  Repos: {cloned} new, {existed} existing" + (f", {len(failed_c)} failed" if failed_c else ""))

    # Install dependencies
    all_dirs = [d for d in os.listdir(nodes_dir) if os.path.isdir(os.path.join(nodes_dir, d))]
    deps = [(d, os.path.join(nodes_dir, d, "requirements.txt"))
            for d in all_dirs if os.path.exists(os.path.join(nodes_dir, d, "requirements.txt"))]
    if deps:
        print(f"  Installing deps for {len(deps)} node(s)...")
        dep_ok, dep_fail = 0, []
        for i, (nd, req) in enumerate(deps, 1):
            r = subprocess.run([pip_exec, "install", "-r", req, "-q"],
                               capture_output=True, text=True, timeout=600)
            if r.returncode == 0: dep_ok += 1
            else: dep_fail.append(nd)
            print(f"\r  Dependencies: {int(i/len(deps)*100)}% ({i}/{len(deps)})", end="", flush=True)
        print()
        print(f"  Deps: {dep_ok}/{len(deps)} ok" + (f", {len(dep_fail)} failed" if dep_fail else ""))
    print("✅ Custom nodes installed!")


def do_install_single_node(repo_url):
    """Install a single custom node from its repo URL + its requirements."""
    repo_url = repo_url.strip()
    if not repo_url:
        print("❌ Please enter a repo URL!")
        return False
    if not repo_url.startswith("http"):
        print("❌ Invalid URL — must start with https://")
        return False

    nodes_dir = f"{COMFY}/custom_nodes"
    os.makedirs(nodes_dir, exist_ok=True)
    os.chdir(nodes_dir)

    repo_name = repo_url.rstrip("/").split("/")[-1].replace(".git", "")
    print(f"📦 Installing: {repo_name}")
    print(f"   URL: {repo_url}")

    if os.path.exists(repo_name):
        print(f"  ⏭️  Already exists — pulling latest...")
        _sh(f"cd {nodes_dir}/{repo_name} && git pull")
    else:
        print(f"  📥 Cloning...")
        r = subprocess.run(["git", "clone", repo_url], capture_output=True, text=True, timeout=180)
        if r.returncode != 0:
            print(f"  ❌ Clone failed!")
            if r.stderr: print(f"     {r.stderr.strip()}")
            return False
        print(f"  ✅ Cloned successfully")

    # Install requirements if present
    req_file = os.path.join(nodes_dir, repo_name, "requirements.txt")
    if os.path.exists(req_file):
        pip_exec = PIP if os.path.exists(PY) else "pip3"
        print(f"  📦 Installing requirements...")
        r = subprocess.run([pip_exec, "install", "-r", req_file],
                           capture_output=True, text=True, timeout=600)
        if r.returncode == 0:
            print(f"  ✅ Dependencies installed")
        else:
            print(f"  ⚠️  Some dependencies may have failed:")
            for ln in (r.stderr or r.stdout or "").strip().splitlines()[-5:]:
                print(f"     {ln}")
    else:
        print(f"  ℹ️  No requirements.txt found — no extra deps needed")

    print(f"✅ {repo_name} installed!")
    return True


def do_fix_deps():
    """Reinstall/fix dependencies."""
    print("🔧 Fixing dependencies...")
    p = PIP if os.path.exists(PY) else "pip3"
    _sh(f"{p} install -r {COMFY}/requirements.txt -q")
    _sh(f"{p} install piexif gguf spandrel rotary-embedding-torch -q")
    _sh(f'{p} install "rembg[gpu]" -q')
    _sh(f"{p} install rembg -q")
    print("✅ Dependencies fixed!")




def do_fix_sam3():
    """Patch ComfyUI-SAM3 to remove the broken comfy_env dependency.

    The SAM3 node pack ships with an experimental 'comfy_env' package
    that is not pip-installable and causes ModuleNotFoundError on every
    startup.  This function:
      1. Clones (or updates) the repo.
      2. Overwrites __init__.py to directly import from the nodes/
         sub-package (which already has proper NODE_CLASS_MAPPINGS).
      3. Overwrites prestartup_script.py to copy assets without comfy_env.
      4. Overwrites install.py to pip-install the real deps.
      5. pip-installs those deps immediately.
    """
    sam3_dir = f"{COMFY}/custom_nodes/ComfyUI-SAM3"
    p = PIP if os.path.exists(PY) else "pip3"

    # ── 1. Clone if not present ────────────────────────────────────────
    if not os.path.exists(sam3_dir):
        print("📦 Cloning ComfyUI-SAM3...")
        r = subprocess.run(
            ["git", "clone",
             "https://github.com/PozzettiAndrea/ComfyUI-SAM3.git",
             sam3_dir, "-q"],
            capture_output=True, text=True, timeout=180
        )
        if r.returncode != 0:
            print(f"  ❌ Clone failed! {r.stderr.strip()}")
            return False
        print("  ✅ Cloned ComfyUI-SAM3")
    else:
        print("  ℹ️  ComfyUI-SAM3 already cloned, pulling latest...")
        _sh(f"cd {sam3_dir} && git pull -q", True)

    # ── 2. Overwrite __init__.py ────────────────────────────────────────
    print("🔧 Patching __init__.py ...")
    _INIT = (
        '"""ComfyUI-SAM3 — patched __init__.py (comfy_env removed).\n'
        '\n'
        'Direct import from the nodes sub-package (already has proper\n'
        'NODE_CLASS_MAPPINGS / NODE_DISPLAY_NAME_MAPPINGS dicts).\n'
        '"""\n'
        '\n'
        'from nodes import (  # noqa: F401\n'
        '    NODE_CLASS_MAPPINGS,\n'
        '    NODE_DISPLAY_NAME_MAPPINGS,\n'
        ')\n'
        '\n'
        'WEB_DIRECTORY = "./web"\n'
        '__all__ = ["NODE_CLASS_MAPPINGS", "NODE_DISPLAY_NAME_MAPPINGS",'
                   ' "WEB_DIRECTORY"]\n'
    )
    with open(f"{sam3_dir}/__init__.py", "w") as _fh:
        _fh.write(_INIT)

    # ── 3. Overwrite prestartup_script.py ──────────────────────────────
    print("🔧 Patching prestartup_script.py ...")
    _PRESTARTUP = (
        '"""ComfyUI-SAM3 prestartup — patched (comfy_env removed)."""\n'
        'import shutil\n'
        'from pathlib import Path\n'
        '\n'
        'SCRIPT_DIR = Path(__file__).resolve().parent\n'
        'COMFYUI_DIR = SCRIPT_DIR.parent.parent\n'
        '\n'
        '# Copy sample assets into ComfyUI/input/ (skip if already present)\n'
        'src = SCRIPT_DIR / "assets"\n'
        'dst = COMFYUI_DIR / "input"\n'
        'if src.exists() and dst.exists():\n'
        '    for item in src.iterdir():\n'
        '        dest_item = dst / item.name\n'
        '        if not dest_item.exists():\n'
        '            if item.is_dir():\n'
        '                shutil.copytree(str(item), str(dest_item))\n'
        '            else:\n'
        '                shutil.copy2(str(item), str(dest_item))\n'
    )
    with open(f"{sam3_dir}/prestartup_script.py", "w") as _fh:
        _fh.write(_PRESTARTUP)

    # ── 4. Overwrite install.py ─────────────────────────────────────────
    print("🔧 Patching install.py ...")
    _INSTALL = (
        '"""ComfyUI-SAM3 install — patched (comfy_env removed)."""\n'
        'import subprocess, sys\n'
        '\n'
        'DEPS = [\n'
        '    "safetensors>=0.4.0", "timm", "ftfy", "regex", "iopath",\n'
        '    "huggingface_hub", "einops", "opencv-python", "psutil",\n'
        '    "scikit-image", "pillow", "tqdm", "typing_extensions",\n'
        '    "requests", "transformers",\n'
        ']\n'
        '\n'
        'subprocess.run([sys.executable, "-m", "pip", "install", "-q"]'
                       ' + DEPS, check=True)\n'
        'print("[SAM3] Dependencies installed.")\n'
    )
    with open(f"{sam3_dir}/install.py", "w") as _fh:
        _fh.write(_INSTALL)

    # ── 5. pip-install the actual SAM3 runtime deps ─────────────────────
    print("📦 Installing SAM3 pip dependencies...")
    _sam3_deps = [
        "safetensors>=0.4.0", "timm", "ftfy", "regex", "iopath",
        "huggingface_hub", "einops", "opencv-python", "psutil",
        "scikit-image", "pillow", "tqdm", "typing_extensions",
        "requests", "transformers",
    ]
    _deps_str = " ".join(f'"{d}"' for d in _sam3_deps)
    _sh(f"{p} install -q {_deps_str}")

    print("✅ ComfyUI-SAM3 patched — comfy_env no longer needed!")
    return True



def do_archive():
    """Archive ComfyUI + venv + zrok2 into a snapshot."""
    VENV_FIX = (
        '#!/usr/bin/env python3\n'
        'import os, re\n'
        'from pathlib import Path\n'
        'def repair():\n'
        '    V = Path("/kaggle/working/venv")\n'
        '    if not V.exists(): return\n'
        '    print("Repairing venv...")\n'
        '    cfg = V / "pyvenv.cfg"\n'
        '    if cfg.exists(): cfg.write_text(re.sub(r"home = .*", "home = /kaggle/working/venv/bin", cfg.read_text()))\n'
        '    for a, p, r in [\n'
        '        ("activate", r\'VIRTUAL_ENV="[^"]*"\', \'VIRTUAL_ENV="/kaggle/working/venv"\'),\n'
        '        ("activate.csh", r\'setenv VIRTUAL_ENV "[^"]*"\', \'setenv VIRTUAL_ENV "/kaggle/working/venv"\'),\n'
        '        ("activate.fish", r\'set -gx VIRTUAL_ENV "[^"]*"\', \'set -gx VIRTUAL_ENV "/kaggle/working/venv"\'),\n'
        '    ]:\n'
        '        f = V / "bin" / a\n'
        '        if f.exists(): f.write_text(re.sub(p, r, f.read_text()))\n'
        '    fixed = 0\n'
        '    for s in (V / "bin").iterdir():\n'
        '        if not s.is_file(): continue\n'
        '        try:\n'
        '            lines = s.read_text(errors="ignore").splitlines(True)\n'
        '            if lines and lines[0].startswith("#!") and "python" in lines[0]:\n'
        '                lines[0] = "#!/kaggle/working/venv/bin/python\\n"\n'
        '                s.write_text("".join(lines)); fixed += 1\n'
        '        except: pass\n'
        '    print(f"Fixed {fixed} shebangs. Done!")\n'
        'if __name__ == "__main__": repair()\n'
    )
    folders = [(f"{WORK}/ComfyUI","ComfyUI"), (f"{WORK}/venv","venv"), (f"{WORK}/zrok2","zrok2")]
    existing = [(p, n) for p, n in folders if os.path.exists(p)]
    if not existing:
        print("❌ Nothing to archive!")
        return

    print("📦 Calculating sizes...")
    total = 0
    for path, name in existing:
        sz = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fn in os.walk(path) for f in fn)
        total += sz
        print(f"  • {name}: {sz/(1024**3):.2f} GB")
    print(f"  Total: {total/(1024**3):.2f} GB")

    fix_file = f"{WORK}/_venv_repair.py"
    with open(fix_file, "w") as f:
        f.write(VENV_FIX)

    print(f"💾 Creating {SNAP_OUT} (this may take several minutes)...")
    with tarfile.open(SNAP_OUT, "w:gz") as tar:
        for path, arcname in existing:
            print(f"  Adding {arcname}...")
            tar.add(path, arcname=arcname)
        tar.add(fix_file, arcname="_venv_repair.py")

    if os.path.exists(fix_file):
        os.remove(fix_file)

    if os.path.exists(SNAP_OUT):
        final = os.path.getsize(SNAP_OUT) / (1024**3)
        print(f"✅ Archive complete! {final:.2f} GB")
        print(f"📉 Compression: {final/max(total/(1024**3),0.01)*100:.1f}%")
        print("📋 Next: 'Save Version' → 'Quick Save' in Kaggle")
    else:
        print("❌ Archive creation failed!")


def do_cleanup():
    """Remove setup files to free space."""
    print("🗑️  Cleaning up...")
    for p in ["ComfyUI", "pinggy.py", "temp-models", "venv", "zrok2", "req.txt", ".virtual_documents"]:
        fp = f"{WORK}/{p}"
        if os.path.exists(fp):
            if os.path.isdir(fp): shutil.rmtree(fp, ignore_errors=True)
            else: os.remove(fp)
            print(f"  Removed {p}")
    _sh("df -h /kaggle/working | tail -1")
    print("✅ Cleanup complete!")


def do_start_comfyui(port="8180", zrok_name=""):
    """Start ComfyUI with zrok2 tunneling. BLOCKING call."""
    py = PY if os.path.exists(PY) else "/usr/bin/python3"
    print(f"🚀 STARTING COMFYUI")
    print(f"🐍 Python: {py} | 🌐 Port: {port}")
    if not os.path.exists(ZROK2):
        print("❌ zrok2 binary not found!")
        return
    _sh(f"chmod a+x {ZROK2}", True)

    share = f"{ZROK2} share public localhost:{port}"
    if zrok_name:
        share += f" -n public:{zrok_name}"
    filt = r"| sed -u 's/\([a-z0-9]*\.shares\.zrok\.io\)/https:\/\/\1/g'"
    cmd = f"{py} {COMFY}/main.py --listen 0.0.0.0 --port {port} & ( {share} --headless || {share} ) {filt}"

    print("⏳ Launching... (zrok URL will appear below)")
    print("─" * 50)
    _SHOW = ("zrok.io", "https://", "share", "zrok", "started", "Starting",
             "error", "Error", "ERROR", "traceback", "Traceback",
             "warning", "Warning", "failed", "Failed", "listening", "Listening",
             "port", "Port", "✅", "❌", "🌐", "🚀")
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in iter(proc.stdout.readline, ""):
            if any(kw in line for kw in _SHOW):
                print(line, end="", flush=True)
    except KeyboardInterrupt:
        proc.terminate()
    finally:
        proc.stdout.close()
        proc.wait()


print(f"✅ Engine loaded! {len(ALL_SECTIONS)} model sections, {sum(len(s) for _,_,s in ALL_SECTIONS)} models, {_n_total} custom nodes ({_n_active} active)")
print("   Default: only Flux 2 Klein 4B models are pre-selected")
print("   → Run the next cell to display the Control Panel")


In [ ]:
# ============================================================
# 🎛️ CONTROL PANEL — Run this cell to display the interactive UI
# ============================================================

import ipywidgets as w
from IPython.display import display, clear_output
from collections import OrderedDict

# ──────────────── STYLING ────────────────
S = {"description_width": "initial"}
BTN_L = w.Layout(width="220px", height="36px")
OUT_L = w.Layout(max_height="400px", overflow_y="auto", border="1px solid #555", width="100%")
WIDE  = w.Layout(width="100%")
TAB_L = w.Layout(width="100%", padding="12px")

def _btn(label, color="#4a90d9", icon="play"):
    return w.Button(description=f"  {label}", layout=BTN_L,
                    style={"button_color": color, "font_weight": "bold"}, icon=icon)

# ════════════════════════════════════════════════════════════
#  TAB 1: SETUP
# ════════════════════════════════════════════════════════════

flow = w.ToggleButtons(
    options=["🆕 First Time Setup", "🔄 Returning User"],
    value="🔄 Returning User", style=S,
    layout=w.Layout(width="100%"),
)
token_w = w.Text(value="jp9EY3XaOGfm", description="Zrok2 Token:", style=S,
                 layout=w.Layout(width="420px"))
zname_w = w.Text(value="", description="Static URL:", style=S,
                 placeholder="(optional, e.g. comfyui)", layout=w.Layout(width="420px"))
reserved_action_w = w.Dropdown(
    options=["Use Existing", "Delete Existing & Create/Use Requested"],
    value="Use Existing",
    description="Reserved Action:",
    style=S, layout=w.Layout(width="520px"))

btn_extract = _btn("Extract Snapshot", "#5cb85c", "archive")
btn_setup   = _btn("Full Setup", "#d9534f", "cog")
btn_zrok    = _btn("Enable Zrok2", "#f0ad4e", "link")
setup_out   = w.Output(layout=OUT_L)

# Show/hide based on flow
first_box = w.VBox([
    w.HTML("<p style='color:#ccc'>Installs ComfyUI, venv, deps, zrok2. Takes ~15 min.</p>"),
    btn_setup,
])
return_box = w.VBox([
    w.HTML("<p style='color:#ccc'>Unpacks your saved snapshot. Takes ~2 min.</p>"),
    btn_extract,
])

def _flow_change(change):
    is_first = "First" in change["new"]
    first_box.layout.display = "" if is_first else "none"
    return_box.layout.display = "none" if is_first else ""

flow.observe(_flow_change, "value")
_flow_change({"new": flow.value})

def _on_extract(b):
    with setup_out:
        clear_output()
        do_extract_snapshot()

def _on_setup(b):
    with setup_out:
        clear_output()
        do_full_setup()

def _on_zrok(b):
    with setup_out:
        clear_output()
        chosen = do_enable_zrok(token_w.value, "public", zname_w.value, reserved_action_w.value)
        if chosen:
            zname_w.value = chosen

btn_extract.on_click(_on_extract)
btn_setup.on_click(_on_setup)
btn_zrok.on_click(_on_zrok)

setup_tab = w.VBox([
    w.HTML("<h3>🔧 Setup</h3><hr>"),
    w.HTML("<b>1. Choose your flow:</b>"),
    flow,
    w.HBox([first_box, return_box]),
    w.HTML("<br><b>2. Configure Zrok2 Tunneling:</b>"),
    token_w, zname_w, reserved_action_w,
    btn_zrok,
    w.HTML("<br><b>Output:</b>"),
    setup_out,
], layout=TAB_L)

# ════════════════════════════════════════════════════════════
#  TAB 2: MODELS
# ════════════════════════════════════════════════════════════

storage_html = w.HTML()
model_cbs = {}  # (section_key, index) → Checkbox widget

def _update_storage(*args):
    """Recalculate estimated download size from checked models."""
    total = 0.0
    for (key, idx), cb in model_cbs.items():
        if cb.value:
            sec = next(s for k, _, s in ALL_SECTIONS if k == key)
            total += sec[idx]["gb"]
    color = "#5cb85c" if total < 40 else ("#f0ad4e" if total < 55 else "#d9534f")
    pct = min(total / 60 * 100, 100)
    storage_html.value = (
        f'<div style="margin:8px 0">'
        f'<b>📊 Estimated Download: <span style="color:{color}">{total:.1f} GB</span> / 60 GB available</b>'
        f'<div style="background:#333;border-radius:4px;height:18px;margin-top:4px">'
        f'<div style="background:{color};width:{pct:.0f}%;height:100%;border-radius:4px;'
        f'transition:width 0.3s"></div></div></div>'
    )

# Build accordion sections
section_panels = []
for sec_key, sec_label, sec_models in ALL_SECTIONS:
    # Group models by "grp" field
    groups = OrderedDict()
    for i, m in enumerate(sec_models):
        g = m.get("grp", "General")
        if g not in groups:
            groups[g] = []
        # Create checkbox
        sz = f"~{m['gb']:.1f} GB" if m["gb"] >= 0.1 else f"~{int(m['gb']*1024)} MB"
        cb = w.Checkbox(value=m["on"], description=f"{m['name']}  ({sz})",
                        indent=False, style=S, layout=WIDE)
        cb.observe(_update_storage, "value")
        model_cbs[(sec_key, i)] = cb
        groups[g].append(cb)

    # Build section content with group headers
    parts = []
    for g, cbs in groups.items():
        if len(groups) > 1:
            parts.append(w.HTML(f"<b style='color:#999;font-size:0.9em'>{g}</b>"))
        parts.extend(cbs)
    section_panels.append((sec_label, w.VBox(parts, layout=w.Layout(padding="4px 0"))))

model_acc = w.Accordion(children=[p for _, p in section_panels])
for i, (label, _) in enumerate(section_panels):
    model_acc.set_title(i, label)
model_acc.selected_index = None  # all collapsed

_update_storage()  # initial calculation

btn_dl = _btn("Download Models", "#5cb85c", "download")
models_out = w.Output(layout=OUT_L)

def _on_download(b):
    with models_out:
        clear_output()
        selected = []
        for (key, idx), cb in model_cbs.items():
            if cb.value:
                sec = next(s for k, _, s in ALL_SECTIONS if k == key)
                selected.append(sec[idx])
        do_download_models(selected)

btn_dl.on_click(_on_download)

# ── Download Custom Model by URL ──
custom_model_url_w = w.Text(
    value="", description="URL:",
    placeholder="https://huggingface.co/…/model.safetensors",
    style=S, layout=w.Layout(width="100%"))
custom_model_name_w = w.Text(
    value="", description="Filename:",
    placeholder="(auto-detected from URL if left empty)",
    style=S, layout=w.Layout(width="100%"))
custom_model_type_w = w.Dropdown(
    options=list(MODEL_TYPE_MAP.keys()),
    value="Checkpoint",
    description="Type:",
    style=S, layout=w.Layout(width="300px"))
btn_dl_custom = _btn("Download", "#0275d8", "download")

def _on_dl_custom(b):
    with models_out:
        clear_output()
        do_download_custom_model(
            custom_model_url_w.value,
            custom_model_name_w.value,
            custom_model_type_w.value)

btn_dl_custom.on_click(_on_dl_custom)

# ── Manage Downloaded Files in /tmp/models ──
manage_type_filter_w = w.Dropdown(
    options=["All Types"] + list(MODEL_TYPE_MAP.keys()) + ["Other"],
    value="All Types",
    description="Filter:",
    style=S, layout=w.Layout(width="320px"))
manage_move_type_w = w.Dropdown(
    options=list(MODEL_TYPE_MAP.keys()),
    value="VAE",
    description="Move To:",
    style=S, layout=w.Layout(width="320px"))
btn_manage_refresh = _btn("Refresh List", "#5bc0de", "refresh")
btn_manage_delete  = _btn("Delete Selected", "#d9534f", "trash")
btn_manage_move    = _btn("Move Selected", "#f0ad4e", "exchange")
manage_status_html = w.HTML()
manage_select_w = w.SelectMultiple(
    options=[],
    rows=10,
    layout=w.Layout(width="100%"),
    description="Files:",
    style=S)


def _refresh_manage_list(*args):
    items = scan_tmp_models()
    filt = manage_type_filter_w.value
    if filt != "All Types":
        items = [x for x in items if x.get("type") == filt]

    total_b = sum(x.get("bytes", 0) for x in items)
    manage_status_html.value = (
        f"<span style='color:#aaa'>Found <b>{len(items)}</b> item(s), total <b>{_human_mb(total_b)}</b>. "
        f"Select one or more files/folders to move or delete.</span>"
    )

    opts = []
    for x in items:
        rel = x.get("rel", "")
        t = x.get("type", "Other")
        sz = _human_mb(x.get("bytes", 0)) if not x.get("is_dir") else "folder"
        label = f"[{t}] {rel} ({sz})"
        opts.append((label, rel))
    manage_select_w.options = opts


def _on_manage_refresh(b):
    _refresh_manage_list()


def _on_manage_delete(b):
    with models_out:
        clear_output()
        selected = list(manage_select_w.value)
        if not selected:
            print("⚠️ Select files/folders first.")
            return
        delete_tmp_models(selected)
        _refresh_manage_list()


def _on_manage_move(b):
    with models_out:
        clear_output()
        selected = list(manage_select_w.value)
        if not selected:
            print("⚠️ Select files/folders first.")
            return
        move_tmp_models(selected, manage_move_type_w.value)
        _refresh_manage_list()


btn_manage_refresh.on_click(_on_manage_refresh)
btn_manage_delete.on_click(_on_manage_delete)
btn_manage_move.on_click(_on_manage_move)
manage_type_filter_w.observe(_refresh_manage_list, "value")
_refresh_manage_list()

models_tab = w.VBox([
    w.HTML("<h3>📦 Models</h3>"
           "<p style='color:#aaa'>Check the models you need. Watch the storage bar — stay under 60 GB.</p><hr>"),
    storage_html,
    model_acc,
    w.HTML("<br>"),
    btn_dl,
    w.HTML("<br><b style='color:#ccc'>📎 Download Custom Model by URL:</b>"
           "<p style='color:#777;font-size:0.85em;margin:2px 0 6px'>Paste any download link. "
           "Pick the model type so it lands in the right ComfyUI folder. "
           "Filename is auto-detected if you leave it empty.</p>"),
    custom_model_url_w,
    custom_model_name_w,
    custom_model_type_w,
    btn_dl_custom,
    w.HTML("<hr><b style='color:#ccc'>🗂️ Manage Downloaded Files (User-Friendly):</b>"
           "<p style='color:#777;font-size:0.85em;margin:2px 0 6px'>Select files in your current tmp storage and move them by model type (e.g., UNet → VAE) or delete permanently.</p>"),
    manage_status_html,
    manage_type_filter_w,
    manage_select_w,
    w.HBox([btn_manage_refresh, manage_move_type_w]),
    w.HBox([btn_manage_move, btn_manage_delete]),
    w.HTML("<br><b>Output:</b>"),
    models_out,
], layout=TAB_L)

# ════════════════════════════════════════════════════════════
#  TAB 3: CUSTOM NODES
# ════════════════════════════════════════════════════════════

node_cbs = []  # list of (Checkbox, url)
for n in NODES_CATALOG:
    cb = w.Checkbox(value=n["on"], description=n["name"], indent=False, style=S, layout=WIDE)
    node_cbs.append((cb, n["url"]))

# Split into Core / Optional
n_core = sum(1 for n in NODES_CATALOG if n["on"])
core_box = w.VBox([cb for cb, _ in node_cbs[:n_core]])
opt_box  = w.VBox([cb for cb, _ in node_cbs[n_core:]])

nodes_acc = w.Accordion(children=[core_box, opt_box])
nodes_acc.set_title(0, f"Core Nodes ({n_core} enabled by default)")
nodes_acc.set_title(1, f"Optional Nodes ({len(node_cbs) - n_core} available)")
nodes_acc.selected_index = None

# ── Install Custom Node by URL ──
custom_url_w = w.Text(
    value="", description="Repo URL:",
    placeholder="https://github.com/user/ComfyUI-NodeName.git",
    style=S, layout=w.Layout(width="100%"))
btn_add_node = _btn("Add & Install", "#0275d8", "plus")

btn_nodes    = _btn("Install Nodes", "#5cb85c", "download")
btn_fix_deps = _btn("Fix Dependencies", "#f0ad4e", "wrench")
btn_fix_sam3 = _btn("Fix SAM3 Nodes", "#9b59b6", "puzzle-piece")
nodes_out    = w.Output(layout=OUT_L)

def _on_nodes(b):
    with nodes_out:
        clear_output()
        selected = [url for cb, url in node_cbs if cb.value]
        do_install_nodes(selected)

def _on_fix(b):
    with nodes_out:
        clear_output()
        do_fix_deps()

def _on_fix_sam3(b):
    with nodes_out:
        clear_output()
        do_fix_sam3()

btn_nodes.on_click(_on_nodes)
btn_fix_deps.on_click(_on_fix)
btn_fix_sam3.on_click(_on_fix_sam3)

def _on_add_node(b):
    with nodes_out:
        clear_output()
        do_install_single_node(custom_url_w.value)

btn_add_node.on_click(_on_add_node)

nodes_tab = w.VBox([
    w.HTML("<h3>🔌 Custom Nodes</h3>"
           "<p style='color:#aaa'>Toggle nodes on/off, or add your own by URL below.</p><hr>"),
    nodes_acc,
    w.HTML("<br><b style='color:#ccc'>📎 Add Custom Node by URL:</b>"),
    custom_url_w,
    btn_add_node,
    w.HTML("<hr>"),
    w.HBox([btn_nodes, btn_fix_deps, btn_fix_sam3]),
    w.HTML("<br><b>Output:</b>"),
    nodes_out,
], layout=TAB_L)

# ════════════════════════════════════════════════════════════
#  TAB 4: LAUNCH
# ════════════════════════════════════════════════════════════

btn_archive = _btn("Archive & Save", "#f0ad4e", "archive")
btn_cleanup = _btn("Cleanup", "#d9534f", "trash")
btn_start   = _btn("Start ComfyUI", "#5cb85c", "rocket")
launch_out  = w.Output(layout=w.Layout(max_height="500px", overflow_y="auto",
                                        border="1px solid #555", width="100%"))

def _on_archive(b):
    with launch_out:
        clear_output()
        do_archive()

def _on_cleanup(b):
    with launch_out:
        clear_output()
        do_cleanup()

def _on_start(b):
    with launch_out:
        clear_output()
        do_start_comfyui(zrok_name=zname_w.value)

btn_archive.on_click(_on_archive)
btn_cleanup.on_click(_on_cleanup)
btn_start.on_click(_on_start)

launch_tab = w.VBox([
    w.HTML("<h3>🚀 Launch</h3><hr>"),
    w.HTML("""<div style="margin-bottom:12px">
    <b>First-time users:</b> Archive your setup before launching so you can restore it later.<br>
    <b>Returning users:</b> Skip straight to Start ComfyUI.<br><br>
    <span style="color:#f0ad4e">⚠️ "Start ComfyUI" is a <b>blocking</b> action — the notebook will stay busy
    until you stop the kernel. Use <b>Kernel → Interrupt</b> to stop.</span>
    </div>"""),
    w.HBox([btn_archive, btn_cleanup]),
    w.HTML("<br>"),
    btn_start,
    w.HTML("<br><b>Output:</b>"),
    launch_out,
], layout=TAB_L)

# ════════════════════════════════════════════════════════════
#  ASSEMBLE PANEL
# ════════════════════════════════════════════════════════════

tabs = w.Tab(children=[setup_tab, models_tab, nodes_tab, launch_tab])
tabs.set_title(0, "🔧 Setup")
tabs.set_title(1, "📦 Models")
tabs.set_title(2, "🔌 Nodes")
tabs.set_title(3, "🚀 Launch")

panel = w.VBox([
    w.HTML("""<div style="background:linear-gradient(135deg,#1a1a2e,#16213e);
    padding:16px 20px;border-radius:8px;margin-bottom:8px">
    <h2 style="margin:0;color:#e94560">🎛️ ComfyUI Control Panel</h2>
    <p style="margin:4px 0 0;color:#a7a7a7">
    Interactive setup for ComfyUI on Kaggle &mdash; no more commenting/uncommenting!
    </p></div>"""),
    tabs,
], layout=w.Layout(width="100%"))

display(panel)
